# Instacart reorder prediction — v2: timing feature

Follow-up experiment to the main model notebook (v1), part of the master's thesis **"Predictive Analytics using SQL and Machine Learning: The case of Instacart"**.

**Question:** does one additional timing feature improve the reorder model?

**Design:** everything is kept identical to v1 (same 20% user sample, same user-level split, same tuned XGBoost parameters) and **only one feature is added**. Any difference in the results can therefore be attributed to that feature.

**The new feature**

```
user_product_days_since_last_at_target
    = user_product_days_since_last_order + target_days_since_prior_order
```

The number of days since the user last bought the product, measured **at the moment of the order being predicted**. `user_product_days_since_last_order` counts days up to the user's previous order, and `target_days_since_prior_order` adds the gap until the target order. A tree model can only approximate this sum through many splits on the two separate columns, so it is given to the model directly.

**Workflow**

load data + new feature → user-level split → X / y → train v2 → compare v1 vs. v2 on the **validation** set

This notebook stops at the validation comparison. The test set is created (so the split matches v1) but is not used for evaluation here.

**Requirements** (in the same folder, all produced by the v1 notebook)
- `ml_sample_20_clean.parquet`: cleaned 20% user sample
- `best_params.json`: tuned v1 hyperparameters
- `instacart_utils.py`: project helper functions

**Files produced:** `xgb_final_v2.json` (trained v2 model)

## 1. Setup
Imports, project helpers and the constants shared by the whole notebook.

In [ ]:
# Reload instacart_utils.py automatically when it is edited (no kernel restart needed)
%load_ext autoreload
%autoreload 2
 
import json 
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,             # used to find the F1-optimal threshold
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)
 
# Project helper functions, shared with v1 (see instacart_utils.py)
from instacart_utils import (
    mean_f1_per_order,       # F1 per user's next order, averaged across users
    per_user_predictions,    # per-user basket selection by expected F1
    basket_summary,          # predicted vs. actual basket statistics
    none_precision,          # how often an empty ("None") basket prediction is correct
    feature_group,           # maps a feature name to its group (user, product, ...)
)
 
RANDOM_STATE = 42            # same seed as v1 -> same split and comparable models
TARGET = "target_variable"

# Identifiers and the target, excluded from the model features
DROP_COLS = ["user_id", "product_id", "order_id", TARGET]
NEW_FEATURE = "user_product_days_since_last_at_target"

## 2. v1 reference results and the new feature

The v1 validation results are stored as constants, so the comparison in section 6 is explicit and reproducible without re-running v1.

The new feature uses only values from the same row, so it can be computed **before** the split without leaking information between train, validation and test. Every row gets a value (no missing values), ranging from 0 to 365 days with a median of 70.

In [ ]:
# v1 results on the VALIDATION set (reference point for the comparison)
V1_VAL = {
    "AUC-PR": 0.43380,
    "Global F1": 0.4520,
    "Mean F1 per order (single threshold)": 0.3840,
    "Mean F1 per order (per-user selection)": 0.3957,
}
 
# Load the clean 20% sample (produced by the v1 notebook)
df = pd.read_parquet("ml_sample_20_clean.parquet")
 
# New feature: days since the user last bought the product, counted up to the target order.
# It uses only values from the same row, so it can be added before the split without leakage.
df[NEW_FEATURE] = (
    df["user_product_days_since_last_order"] + df["target_days_since_prior_order"]
)

# 80 columns (v1: 79) and the distribution of the new feature
print(df.shape)
print(df[NEW_FEATURE].describe())

(1693889, 80)
count    1.693889e+06
mean     9.774366e+01
std      8.537343e+01
min      0.000000e+00
25%      3.000000e+01
50%      7.000000e+01
75%      1.450000e+02
max      3.650000e+02
Name: user_product_days_since_last_at_target, dtype: float64


## 3. User-level split (identical to v1)

Same two-step `GroupShuffleSplit` and same `random_state` as v1, so exactly the same users end up in train, validation and test. The row counts below match v1 (1,187,806 / 251,739 / 254,344), which confirms that the two versions are trained and evaluated on the same data.

In [ ]:
# User-level split 70 / 15 / 15 with the same random_state as v1 -> the same users in each set
# First split: hold out 15% of users for testing
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_STATE)
train_val_idx, test_idx = next(gss.split(df, groups=df["user_id"]))
train_val = df.iloc[train_val_idx]
test = df.iloc[test_idx]

# Second split: 0.15 / 0.85 ≈ 17.6% of the remaining users -> 15% of all users
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.15 / 0.85, random_state=RANDOM_STATE)
train_idx, val_idx = next(gss2.split(train_val, groups=train_val["user_id"]))
train = train_val.iloc[train_idx]
val = train_val.iloc[val_idx]

# Verify that no user appears in more than one set (no user leakage)
tr, va, te = set(train["user_id"]), set(val["user_id"]), set(test["user_id"])
assert not (tr & va) and not (tr & te) and not (va & te), "User leakage between the subsets!"

# Rows, users and positive-class rate per split
summary = pd.DataFrame({
    "rows": [len(train), len(val), len(test)],
    "users": [len(tr), len(va), len(te)],
    "positive_rate": [train[TARGET].mean(), val[TARGET].mean(), test[TARGET].mean()],
}, index=["train", "val", "test"])
display(summary)
# Check: must be IDENTICAL to v1 (train 1,187,806 / val 251,739 / test 254,344 rows)

,rows,users,positive_rate
train,1187806,18381,0.097534
val,251739,3940,0.098272
test,254344,3940,0.098709


## 4. Features, target and model parameters

- **Features:** 76 columns (v1's 75 + the new feature).
- **Parameters:** the tuned v1 parameters from `best_params.json`, with the same final-model settings as v1 (learning rate 0.03, up to 5,000 trees, early-stopping patience 150). They are deliberately **not re-tuned**, so the comparison isolates the effect of the feature.

In [ ]:
# Features (X) and target (y) for each split
X_train, y_train = train.drop(columns=DROP_COLS), train[TARGET]
X_val,   y_val   = val.drop(columns=DROP_COLS),   val[TARGET]
X_test,  y_test  = test.drop(columns=DROP_COLS),  test[TARGET]
 
print(X_train.shape, X_val.shape, X_test.shape)   # 76 στήλες (v1: 75)
assert NEW_FEATURE in X_train.columns
 
# Parameters: the tuned v1 parameters plus the same final-model settings as v1.
# They are NOT re-tuned, so any difference in results comes from the new feature alone.
with open("best_params.json") as f:
    current_params = json.load(f)
 
final_params = {**current_params,
                "learning_rate": 0.03,
                "n_estimators": 5000,
                "early_stopping_rounds": 150}
print(final_params)
 
 
def train_model(params):
    """Εκπαιδεύει XGBoost στο train με early stopping στο validation."""
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return model

(1187806, 76) (251739, 76) (254344, 76)
{'n_estimators': 5000, 'learning_rate': 0.03, 'max_depth': 4, 'min_child_weight': 10, 'subsample': 0.8, 'colsample_bytree': 0.8, 'gamma': 0, 'reg_lambda': 1, 'scale_pos_weight': 1, 'tree_method': 'hist', 'eval_metric': 'aucpr', 'early_stopping_rounds': 150, 'random_state': 42, 'n_jobs': -1}


## 5. Train v2

Training takes about 15 minutes. The model is saved, and after a kernel restart it can be reloaded with the commented-out line instead of retraining.

Validation AUC-PR: **0.43557** with 1,634 trees (v1: 0.43380 with 1,966 trees).

In [ ]:
## Train v2 and save the model
model_v2 = train_model(final_params)
model_v2.save_model("xgb_final_v2.json")
print(f"v2 val AUC-PR = {model_v2.best_score:.5f} (trees: {model_v2.best_iteration})")

v2 val AUC-PR = 0.43557 (trees: 1634)


## 6. Compare v1 and v2 on the validation set

The three decision strategies from v1 are repeated with v2's predictions:

1. **Global F1 threshold:** one cutoff that maximizes row-level F1
2. **Mean F1 per order threshold:** one cutoff that maximizes F1 averaged over users' orders
3. **Per-user selection:** each user's basket chosen to maximize that user's expected F1

Thresholds are re-tuned on validation for v2, since a new model needs its own cutoffs.

In [ ]:
# Compare v1 and v2 on the VALIDATION set
val_proba = model_v2.predict_proba(X_val)[:, 1]             # P(reorder) for each (user, product) row
val_users = val["user_id"].values
y_val_arr = y_val.values
 
# a) Strategy 1: global threshold that maximizes row-level F1
#    (prec/rec have one more point than thr, so the last one is dropped; 1e-12 avoids division by zero)
prec, rec, thr = precision_recall_curve(y_val_arr, val_proba)
f1_scores = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best_idx = np.argmax(f1_scores)
best_threshold = thr[best_idx]
 
# b) Strategy 2: global threshold that maximizes the mean F1 per order
thresholds = np.arange(0.05, 0.50, 0.01)
order_f1 = [mean_f1_per_order(val_users, y_val_arr, (val_proba >= t).astype(int))
            for t in thresholds]
best_t_order = thresholds[np.argmax(order_f1)]
 
# c) Strategy 3: per-user basket selection (maximizes each user's expected F1)
val_pred_user = per_user_predictions(val_users, val_proba)
val_f1_user = mean_f1_per_order(val_users, y_val_arr, val_pred_user)

# best_score = AUC-PR as computed by XGBoost, the same measure as v1's 0.43380.
# (sklearn's average_precision_score gives a slightly different value; it is used on the test set.)
v2_val = {
    "AUC-PR": model_v2.best_score,
    "Global F1": f1_scores[best_idx],
    "Mean F1 per order (single threshold)": max(order_f1),
    "Mean F1 per order (per-user selection)": val_f1_user,
}

# Side-by-side table: diff > 0 means v2 is better
comparison = pd.DataFrame({"v1": V1_VAL, "v2": v2_val})
comparison["diff"] = comparison["v2"] - comparison["v1"]
display(comparison.round(4))
 
print(f"v2 thresholds: global F1 = {best_threshold:.4f} | mean F1 per order = {best_t_order:.2f}")

# Basket-level statistics for v2 (same table as in the v1 notebook)
display(pd.DataFrame({
    "v2 single threshold": basket_summary(val_users, y_val_arr, (val_proba >= best_t_order).astype(int)),
    "v2 per-user": basket_summary(val_users, y_val_arr, val_pred_user),
}).round(2))

# Share of users given an empty ("None") basket who actually reordered nothing
print(f"v2 'None' precision (per-user): {none_precision(val_users, y_val_arr, val_pred_user):.2%}")

,v1,v2,diff
AUC-PR,0.4338,0.4356,0.0018
Global F1,0.4520,0.4521,0.0001
Mean F1 per order (single threshold),0.3840,0.3838,-0.0002
Mean F1 per order (per-user selection),0.3957,0.3955,-0.0002


v2 thresholds: global F1 = 0.2366 | mean F1 per order = 0.20


,v2 single threshold,v2 per-user
avg_actual_reorders,6.28,6.28
avg_predicted_reorders,8.87,9.27
users_with_no_predicted_reorders (%),4.49,3.40
users_with_no_actual_reorders (%),6.60,6.60


v2 'None' precision (per-user): 32.09%


### Results

| Validation metric | v1 | v2 | Difference |
|---|---|---|---|
| AUC-PR | 0.4338 | 0.4356 | +0.0018 |
| Global F1 | 0.4520 | 0.4521 | +0.0001 |
| Mean F1 per order (single threshold) | 0.3840 | 0.3838 | −0.0002 |
| Mean F1 per order (per-user selection) | 0.3957 | 0.3955 | −0.0002 |

- **Ranking quality improves slightly:** AUC-PR rises by 0.0018, about as much as the entire hyperparameter tuning gained in v1 (~0.0015).
- **Basket quality is unchanged:** the decision metrics move by at most 0.0002, so the better ranking does not translate into better baskets on validation.
- Basket behaviour is very close to v1: thresholds 0.2366 / 0.20 (v1: 0.2382 / 0.21), 9.27 predicted products per user with per-user selection (v1: 9.31), and "None" precision 32.1% (v1: 32.4%).